팀장님 피드백 반영 수정본

In [1]:
import torch
import torch.nn as nn

# [부품 1] 정보를 잃어버리지 않는 베테랑 탐정 (RFB)
class ResidualFrequencyBlock(nn.Module):
    def __init__(self, ch):
        super(ResidualFrequencyBlock, self).__init__()
       # 입출력 채널을 동일하게 설정 (ch -> ch)
        self.conv_path = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(ch),
            nn.ReLU(),
            nn.Conv2d(ch, ch, kernel_size=3, padding=1)
            # 마지막 BN, ReLU 제거 -> 날카로운 선(고주파) 보존!
        )
        self.shortcut = nn.Identity()

    def forward(self, x):
        # 공부한 내용 + 원래 내용을 그대로 합침
        return self.conv_path(x) + self.shortcut(x)

# [부품 2] 한 단계를 처리하는 연구소 (Branch)
class FrequencyBranch(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch, s1=2, s2=2):
        super().__init__()
        # 1세트: 똑똑하게 줄이기(Stride) -> 정밀 관찰(RFB)
        self.step1 = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, kernel_size=3, stride=s1, padding=1),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(),
            ResidualFrequencyBlock(mid_ch)
        )
        # 2세트: 똑똑하게 줄이기(Stride) -> 정밀 관찰(RFB)
        self.step2 = nn.Sequential(
            nn.Conv2d(mid_ch, out_ch, kernel_size=3, stride=s2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            ResidualFrequencyBlock(out_ch)
        )

    def forward(self, x):
        return self.step2(self.step1(x))

# [부품 3] 전체를 관리하는 사령탑 (Extractor)
class FrequencyFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # L1은 큰 종이(112)니까 두 번 점프해서 28로! (s1=2, s2=2)
        self.l1_branch = FrequencyBranch(9, 18, 36, s1=2, s2=2)

        # L2는 이미 작은 종이(56)니까 한 번만 점프해서 28로! (s1=2, s2=1)
        self.l2_branch = FrequencyBranch(9, 18, 36, s1=2, s2=1)

        # 최종 보고서 요약 (72채널 -> 144채널)
        self.final_conv = nn.Sequential(
            nn.Conv2d(72, 144, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(144),
            nn.ReLU()
        )

    def forward(self, l1_input, l2_input):
        feat1 = self.l1_branch(l1_input) # 결과: 28x28
        feat2 = self.l2_branch(l2_input) # 결과: 28x28

        # 옆으로 찰떡같이 붙이기
        combined = torch.cat([feat1, feat2], dim=1)

        # 마지막 요약본 만들기
        return self.final_conv(combined)

# [성공 체크] 잘 작동하는지 확인해보기
if __name__ == "__main__":
    model = FrequencyFeatureExtractor()
    mock_l1 = torch.randn(1, 9, 112, 112)
    mock_l2 = torch.randn(1, 9, 56, 56)

    output = model(mock_l1, mock_l2)
    print(f"성공! 최종 보고서 크기: {output.shape}")
    # [1, 144, 14, 14]가 나오면 통과입니다!

성공! 최종 보고서 크기: torch.Size([1, 144, 14, 14])
